# Exp2 → Exp3 Prediction (Quick Run)

This notebook cell uses Exp2 Pass@1 to predict Exp3 (until-correct) success and expected attempts,
using the baseline formulas (q = 1 - (1 - p)^k, A = [1 - (1 - p)^k] / p). Finite-population correction
is optional and controlled via POOL_SIZE.


In [ ]:
# Configuration
RESULTS_DIR = './results/'            # root of results tree
K = 3                               # Exp3: max attempts per type (set to your k)
POOL_SIZE = 20                     # None => i.i.d. baseline; set integer N for finite-pool correction
PROVIDER = 'openai'                      # e.g., 'openai' or None for all
MODEL = 'gpt-5'                         # e.g., 'openai/gpt-5' or None for all
CALIBRATE = False                    # set True to fit simple calibration if Exp3 observations exist

In [9]:
from exp2_to_exp3_predict import (
    predict_q_from_exp2, predict_A_from_exp2,
    calibrate_logit_linear, apply_calibration
)
from visualize_results import CAPTCHAVisualizer
import pandas as pd
import numpy as np

viz = CAPTCHAVisualizer(results_dir=RESULTS_DIR)
if viz.data.empty:
    raise RuntimeError('No results found under RESULTS_DIR')

df = viz.data.copy()
if PROVIDER:
    df = df[df['provider'] == PROVIDER]
if MODEL:
    df = df[df['provider_model'] == MODEL]

# Exp2 summary
exp2 = df[df['experiment'] == 'exp2'].copy()
if exp2.empty:
    raise RuntimeError('Exp2 results not found in current selection')
if 'n' not in exp2.columns:
    exp2['n'] = 1
exp2_grp = exp2.groupby(
    ['provider','model','provider_model','task_type'],
    as_index=False
).agg(
    n=('n','max'),
    p_hat=('pass','mean')
)

# Exp3 observations (optional validation)
exp3 = df[df['experiment'] == 'exp3'].copy()
if not exp3.empty:
    exp3_obs = exp3.groupby(
        ['provider','model','provider_model','task_type'],
        as_index=False
    ).agg(
        q_obs=('pass','mean'),
        A_obs=('avg_attempts','mean')
    )
else:
    exp3_obs = pd.DataFrame(
        columns=['provider','model','provider_model','task_type','q_obs','A_obs']
    )

rows = []
for _, r in exp2_grp.iterrows():
    n = int(r['n'])
    p_hat = float(r['p_hat'])

    # 难度参数 λ = -ln(1 - p)，使用裁剪后的 p 避免 log(0)
    p_clipped = float(np.clip(p_hat, 1e-9, 1.0 - 1e-9))
    lam = -np.log(1.0 - p_clipped)

    # 基线 i.i.d. 形式：q_iid(k) = 1 - exp(-k λ)
    q_iid = 1.0 - np.exp(-K * lam)

    # 最终预测（如果设置了 POOL_SIZE，会包含有限题库修正）
    q_pred = predict_q_from_exp2(p_hat, n, K, N_pool=POOL_SIZE)
    A_pred = predict_A_from_exp2(p_hat, n, K, N_pool=POOL_SIZE)

    rows.append([
        r['provider'], r['model'], r['provider_model'], r['task_type'],
        n, p_hat, lam, q_iid, q_pred, A_pred
    ])

pred_df = pd.DataFrame(
    rows,
    columns=[
        'provider','model','provider_model','task_type',
        'n','p_hat','lambda','q_iid','q_pred','A_pred'
    ]
)
merged = pred_df.merge(
    exp3_obs,
    on=['provider','model','provider_model','task_type'],
    how='left'
)

if CALIBRATE and not exp3_obs.empty:
    valid = merged.dropna(subset=['q_pred','q_obs'])
    if len(valid) >= 3:
        a, b = calibrate_logit_linear(
            valid['q_pred'].to_numpy(), valid['q_obs'].to_numpy()
        )
        merged['q_pred_cal'] = apply_calibration(
            merged['q_pred'].to_numpy(), a, b
        )
        print(f'Calibration fitted: a={a:.3f}, b={b:.3f}')
    else:
        print('Not enough pairs to calibrate; skipping')

display(merged.sort_values(['provider_model','task_type']).head(20))
merged.to_csv('./exp2_to_exp3_predictions_nb.csv', index=False)
print('Saved to ./exp2_to_exp3_predictions_nb.csv')



Scanning experiment result directories...
[LOADED] exp1/gemini/gemini-2.5-pro (18 records)
[LOADED] exp1/gemini/gemini-2.5-flash (18 records)
[LOADED] exp1/fireworks/accounts_fireworks_models_qwen3-vl-235b-a22b-instruct (18 records)
[LOADED] exp1/anthropic/claude-sonnet-4-5 (18 records)
[LOADED] exp1/openai/gpt-5 (18 records)
[LOADED] exp1/openai/gpt-5-chat-latest (18 records)
[LOADED] exp3/gemini/gemini-2.5-pro (18 task types, converted from Exp3 format)
[LOADED] exp3/gemini/gemini-2.5-flash (18 task types, converted from Exp3 format)
[LOADED] exp3/fireworks/accounts_fireworks_models_qwen3-vl-235b-a22b-instruct (18 task types, converted from Exp3 format)
[LOADED] exp3/anthropic/claude-sonnet-4-5 (18 task types, converted from Exp3 format)
[LOADED] exp3/openai/gpt-5 (17 task types, converted from Exp3 format)
[LOADED] exp3/openai/gpt-5-chat-latest (18 task types, converted from Exp3 format)
[LOADED] exp4/gemini/gemini-2.5-flash (4 records)
[LOADED] exp4/openai/gpt-5 (4 records)
[LOADED

,provider,model,provider_model,task_type,n,p_hat,lambda,q_iid,q_pred,A_pred,q_obs,A_obs
0,gemini,gemini-2.5-flash,gemini/gemini-2.5-flash,Bingo,20,0.900,2.302585e+00,9.990000e-01,1.000000,1.894737,1.0,1.0
1,gemini,gemini-2.5-flash,gemini/gemini-2.5-flash,Click_Order,10,0.000,1.000000e-09,3.000000e-09,0.000000,0.000000,0.0,10.0
2,gemini,gemini-2.5-flash,gemini/gemini-2.5-flash,Connect_Icon,20,0.350,4.307829e-01,7.253750e-01,0.749123,0.939474,1.0,2.0
3,gemini,gemini-2.5-flash,gemini/gemini-2.5-flash,Coordinates,18,0.667,1.099613e+00,9.630740e-01,0.969298,1.539474,1.0,2.0
4,gemini,gemini-2.5-flash,gemini/gemini-2.5-flash,Dart_Count,20,0.650,1.049822e+00,9.571250e-01,0.969298,1.539474,1.0,4.0
5,gemini,gemini-2.5-flash,gemini/gemini-2.5-flash,Dice_Count,11,0.000,1.000000e-09,3.000000e-09,0.000000,0.000000,0.0,10.0
6,gemini,gemini-2.5-flash,gemini/gemini-2.5-flash,Geometry_Click,15,0.000,1.000000e-09,3.000000e-09,0.000000,0.000000,0.0,10.0
7,gemini,gemini-2.5-flash,gemini/gemini-2.5-flash,Image_Matching,19,0.684,1.152013e+00,9.684455e-01,0.982456,1.621053,1.0,1.0
8,gemini,gemini-2.5-flash,gemini/gemini-2.5-flash,Image_Recognition,20,0.850,1.897120e+00,9.966250e-01,0.999123,1.834211,1.0,1.0
9,gemini,gemini-2.5-flash,gemini/gemini-2.5-flash,Misleading_Click,10,0.800,1.609438e+00,9.920000e-01,0.996491,1.768421,1.0,1.0


Saved to ./exp2_to_exp3_predictions_nb.csv
